# 04 — Embeddings + FAISS Index
Generates embeddings with `all-MiniLM-L6-v2` and builds a FAISS index.

In [ ]:
!pip install sentence-transformers faiss-cpu -q

In [ ]:
import json
import numpy as np
import faiss
from pathlib import Path
from sentence_transformers import SentenceTransformer

CHUNKS_PATH  = Path('data/processed/chunks.jsonl')
INDEX_DIR    = Path('vector_db/faiss_index')
INDEX_DIR.mkdir(parents=True, exist_ok=True)

MODEL_NAME   = 'sentence-transformers/all-MiniLM-L6-v2'
BATCH_SIZE   = 256

def load_jsonl(path):
    with open(path, 'r', encoding='utf-8') as f:
        return [json.loads(line) for line in f if line.strip()]

chunks = load_jsonl(CHUNKS_PATH)
print(f'Loaded {len(chunks):,} chunks')

In [ ]:
# ── Load model ─────────────────────────────────────────────────────────────
model  = SentenceTransformer(MODEL_NAME)
texts  = [c['text'] for c in chunks]
metas  = [{k: v for k, v in c.items() if k != 'text'} for c in chunks]

print('Encoding embeddings …')
embeddings = model.encode(
    texts,
    batch_size=BATCH_SIZE,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True,
)
print(f'Embeddings shape: {embeddings.shape}')

In [ ]:
# ── Build FAISS IndexFlatIP (cosine on normalized vecs) ────────────────────
dim   = embeddings.shape[1]
index = faiss.IndexFlatIP(dim)
index.add(embeddings.astype(np.float32))
print(f'FAISS index: {index.ntotal:,} vectors')

In [ ]:
# ── Save index + metadata ──────────────────────────────────────────────────
faiss.write_index(index, str(INDEX_DIR / 'index.faiss'))

with open(INDEX_DIR / 'metadata.jsonl', 'w', encoding='utf-8') as f:
    for m in metas:
        f.write(json.dumps(m, ensure_ascii=False) + '\n')

print(f'Saved FAISS index → {INDEX_DIR}/index.faiss')
print(f'Saved metadata    → {INDEX_DIR}/metadata.jsonl')